# Resources Tab

> Worker CPU, RAM, and GPU resource usage display.

In [ ]:
#| default_exp components.tabs.resources_tab

In [ ]:
#| export
from typing import Optional

from fasthtml.common import Div, Span, Progress, FT

from cjm_fasthtml_daisyui.components.feedback.progress import progress as progress_cls, progress_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui, bg_dui
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, font_family
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, items, justify, gap
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_job_monitor.models import ResourceSnapshot

## Helpers

In [ ]:
#| export
def _render_stat_row(
    label: str,          # Stat label (e.g., 'CPU')
    value_text: str,     # Formatted value (e.g., '45.2%')
    bar_pct: Optional[int] = None,   # Progress bar percentage (0-100)
    bar_color: str = '',             # DaisyUI progress color class
) -> FT:  # Stat row element
    """Render a single stat row with label, value, and optional progress bar."""
    children = [
        Div(
            Span(label, cls=combine_classes(font_size.xs, text_dui.base_content)),
            Span(value_text, cls=combine_classes(font_size.xs, font_weight.semibold, font_family.mono)),
            cls=combine_classes(flex_display, items.center, justify.between),
        ),
    ]
    if bar_pct is not None:
        children.append(
            Progress(
                value=str(max(0, min(100, bar_pct))),
                max="100",
                cls=combine_classes(progress_cls, bar_color, w.full, h(1)),
            )
        )
    return Div(*children, cls=m.b(3))

## render_resources_tab

In [ ]:
#| export
def render_resources_tab(
    resources: Optional[ResourceSnapshot] = None,  # Resource data
) -> FT:  # Resources tab content
    """Render resources tab content."""
    if resources is None:
        return Div(
            Span('No resource data available.', cls=combine_classes(font_size.sm, text_dui.base_content)),
            cls=p(4),
        )

    children = []

    # Worker PID
    children.append(
        Div(
            Span('Worker PID: ', cls=combine_classes(font_size.xs, text_dui.base_content)),
            Span(str(resources.worker_pid), cls=combine_classes(font_size.xs, font_weight.semibold, font_family.mono)),
            cls=m.b(3),
        )
    )

    # CPU
    children.append(_render_stat_row(
        label='CPU',
        value_text=f'{resources.cpu_percent:.1f}%',
        bar_pct=int(resources.cpu_percent),
        bar_color=str(progress_colors.info),
    ))

    # Memory RSS
    children.append(_render_stat_row(
        label='Memory (RSS)',
        value_text=f'{resources.memory_rss_mb:.0f} MB',
    ))

    # GPU section (if available)
    if resources.gpu_memory_mb is not None:
        # GPU name header
        gpu_label = resources.gpu_name or f'GPU {resources.gpu_index}'
        children.append(
            Div(
                Span(gpu_label, cls=combine_classes(font_size.xs, font_weight.semibold, text_dui.base_content)),
                cls=combine_classes(m.t(2), m.b(1)),
            )
        )

        # GPU memory
        gpu_pct = None
        gpu_value = f'{resources.gpu_memory_mb:.0f} MB'
        if resources.gpu_total_mb and resources.gpu_total_mb > 0:
            gpu_pct = int((resources.gpu_memory_mb / resources.gpu_total_mb) * 100)
            gpu_value = f'{resources.gpu_memory_mb:.0f} / {resources.gpu_total_mb:.0f} MB'

        children.append(_render_stat_row(
            label='GPU Memory',
            value_text=gpu_value,
            bar_pct=gpu_pct,
            bar_color=str(progress_colors.secondary),
        ))

        # GPU load
        if resources.gpu_load_percent is not None:
            children.append(_render_stat_row(
                label='GPU Load',
                value_text=f'{resources.gpu_load_percent:.0f}%',
                bar_pct=int(resources.gpu_load_percent),
                bar_color=str(progress_colors.secondary),
            ))

    return Div(*children, cls=p(4))

In [ ]:
# Test render_resources_tab -- no data
result = render_resources_tab(None)
assert 'No resource data' in str(result)
print("Resources tab (no data): OK")

# Test render_resources_tab -- CPU/RAM only
from cjm_fasthtml_job_monitor.models import ResourceSnapshot

snap = ResourceSnapshot(worker_pid=12345, cpu_percent=45.2, memory_rss_mb=1200.5)
result = render_resources_tab(snap)
result_str = str(result)
assert '12345' in result_str
assert '45.2%' in result_str
assert '1200 MB' in result_str  # Rounded
print("Resources tab (CPU/RAM): OK")

# Test render_resources_tab -- with GPU
snap_gpu = ResourceSnapshot(
    worker_pid=12345, cpu_percent=30.0, memory_rss_mb=1500.0,
    gpu_memory_mb=2048.0, gpu_index=0, gpu_name='NVIDIA GeForce RTX 4090',
    gpu_total_mb=24564.0, gpu_load_percent=72.0,
)
result = render_resources_tab(snap_gpu)
result_str = str(result)
assert 'RTX 4090' in result_str
assert '2048' in result_str
assert '24564' in result_str
assert '72%' in result_str
print("Resources tab (with GPU): OK")

Resources tab (no data): OK
Resources tab (CPU/RAM): OK
Resources tab (with GPU): OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()